<a href="https://colab.research.google.com/github/yccccc12/transformer_from_scratch/blob/master/src/transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup: Paths & Configuration

In [1]:
from pathlib import Path
import random
import numpy as np
import torch

# Constant File Path
DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints")

DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.txt"
CORPUS_FILE = DATA_DIR / "corpus.txt"

TOKENIZER_FILE = CHECKPOINT_DIR / "tokenizer.json"
MODEL_FILE = CHECKPOINT_DIR / "transformer.pt"

# Set seed to ensure reproducible
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## 1. Prepare Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

# Shuffle before selecting: the first rows of opus_books are book titles/headers,
# not real sentences, which produced degenerate training data.
dataset = dataset.shuffle(seed=42)

dataset = dataset.select(range(20000))

count = 0

with open(str(TRAIN_FILE), "w", encoding="utf-8") as f:
    for example in dataset:
        translation = example["translation"]

        src = translation["en"].strip()
        tgt = translation["fr"].strip()

        f.write(f"{src}\t{tgt}\n")
        count += 1


print(f"Saved {count} examples to {TRAIN_FILE}")

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

en-fr/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

en-fr/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Saved 20000 examples to data/train.txt


## 2. BPE Tokenizer

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

class BPETokenizer:
    def __init__(self, vocab_size=2000):
        self.tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
        self.tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        self.tokenizer.decoder = ByteLevelDecoder()

        self.trainer = BpeTrainer(
            vocab_size=vocab_size,
            special_tokens=["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
        )

    def train(self, files):
        self.tokenizer.train(files, self.trainer)

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

    def encode(self, text):
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        return self.tokenizer.decode(ids)

    @property
    def vocab_size(self):
        return self.tokenizer.get_vocab_size()

    @property
    def pad_id(self):
        return self.tokenizer.token_to_id("<PAD>")

    @property
    def bos_id(self):
        return self.tokenizer.token_to_id("<BOS>")

    @property
    def eos_id(self):
        return self.tokenizer.token_to_id("<EOS>")

    @property
    def unk_id(self):
        return self.tokenizer.token_to_id("<UNK>")

## 3. Attention

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

"""Single Head Attention"""
class Attention(nn.Module):
    def __init__(self, d_model):
        """
        d_model = dimension of embeddings
        """
        super().__init__()

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.d_model = d_model

    def forward(self, x, mask=None):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Last 2 dimension (-2, -1), d_model = d_Q = d_K = d_V (dimension of V)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.d_model ** 0.5

        if mask is not None:
            scores = scores.masked_fill(mask=mask, value=float("-inf"))

        # Apply softmax across the last dimension (dim=-1, d_model values per row), so each row sums to 1
        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        return output

"""
Multi-Head Attention lets the model look at the same sentence in multiple ways at the same time,
so different heads can learn different relationships between the tokens.
"""
class MultiHeadAttenion(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        d_model = dimension of embeddings
        num_heads = number of attention heads
        """
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Project input into Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # Combine all heads back into d_model
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0) # q.size(0) == k.size(0) == v.size(0)
        q_seq_len = q.size(1)  # q can have a different sequence length from K/V in cross-attention
        k_seq_len = k.size(1)  # k and v have the same sequence length

        Q = self.W_q(q)
        K = self.W_k(k)
        V = self.W_v(v)

        # Split heads
        # [batch, seq_len, d_model] -> [batch, seq_len, heads, head_dim]
        Q = Q.view(batch_size, q_seq_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, k_seq_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, k_seq_len, self.num_heads, self.head_dim)

        # [batch, seq_len, heads, head_dim] -> [batch, heads, seq_len, head_dim]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.head_dim ** 0.5

        if mask is not None:
            scores = scores.masked_fill(mask=mask, value=float('-inf'))

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        # [batch, heads, seq_len, head_dim] -> [batch, seq_len, heads, head_dim]
        output = output.transpose(1, 2)

        # Combine heads
        output = output.contiguous().view(batch_size, q_seq_len, self.d_model)

        output = self.W_o(output)

        return output


## 4. Dataset & Collate Function

In [5]:
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class TranslationDataset(Dataset):
    def __init__(self, filepath, tokenizer, max_len):
        self.data = []

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                src_text, tgt_text = line.split("\t")

                src_ids = tokenizer.encode(src_text)
                tgt_ids = tokenizer.encode(tgt_text)

                if len(src_ids) > max_len:
                    continue

                if len(tgt_ids) + 2 > max_len:
                    continue
                # Add special tokens to target
                tgt_ids = [
                    tokenizer.bos_id,
                    *tgt_ids,
                    tokenizer.eos_id,
                ]

                self.data.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_ids, tgt_ids = self.data[idx]

        src = torch.tensor(src_ids, dtype=torch.long)
        tgt = torch.tensor(tgt_ids, dtype=torch.long)

        # Teacher forcing
        decoder_input = tgt[:-1]
        target_output = tgt[1:]

        return src, decoder_input, target_output


def create_collate_fn(pad_id):

    def collate_fn(batch):
        src, decoder_input, target_output = zip(*batch)

        src = pad_sequence(
            src,
            batch_first=True,
            padding_value=pad_id,
        )

        decoder_input = pad_sequence(
            decoder_input,
            batch_first=True,
            padding_value=pad_id,
        )

        target_output = pad_sequence(
            target_output,
            batch_first=True,
            padding_value=pad_id,
        )

        return src, decoder_input, target_output

    return collate_fn

## 5. Token Embedding

In [6]:
class Token_Embedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        """
        vocab_size: number of different tokens
        embedding_dim: number of values used to represent each token
        """
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

    def forward(self, token_ids):
        """
        token_ids: token IDs representing the input text. Each ID corresponds to a token in the vocabulary.
        """
        return self.embedding(token_ids)


## 6. Feed Forward

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        """
        d_model = dimension of embeddings
        d_ff = d_ff = dimension of inner layer (feed forward neural network)
        """
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)


    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x



## 7. Layer Normalization

In [8]:
"""
Layer Normalization stabilizes and accelerates the training process in deep learning.

Source:
https://www.geeksforgeeks.org/deep-learning/what-is-layer-normalization/

"""
class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = 1e-5

    def forward(self, x):
        mean = torch.mean(x, dim=-1, keepdim=True)
        var = torch.var(x, dim=-1, keepdim=True, correction=0)

        output = ((x - mean) / (var + self.eps) ** 0.5) * self.gamma + self.beta

        return output


## 8. Masks

In [9]:
def create_masks(src, tgt, pad_id):
    batch_size = src.size(0) # B
    src_len = src.size(1)    # S
    tgt_len = tgt.size(1)    # T

    # Source padding mask: We don't want encoder to pay attention to PAD
    # [B, S], False = real token, True = PAD token
    src_padding_mask = (src == pad_id)

    # [B, S] -> [B, 1, S] -> [B, 1, 1, S]
    src_mask = src_padding_mask.unsqueeze(1).unsqueeze(2)

    # Target padding mask
    # [B, T]
    tgt_padding_mask = (tgt == pad_id)

    # [B, T] -> [B, 1, 1, T]
    tgt_padding_mask = tgt_padding_mask.unsqueeze(1).unsqueeze(2)


    # Causal mask: Used by decoder self-attention, mask future token
    # True = Block, False = Allow
    causal_mask = torch.triu(
        torch.ones(tgt_len, tgt_len, dtype=torch.bool, device=tgt.device),
        diagonal=1
    )

    # [T, T] -> [1, 1, T, T]
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

    # Combine target mask
    # Decoder don't look at PAD tokens and future tokens
    # [B, 1, T, T], True if either conditions = Block
    tgt_mask = tgt_padding_mask | causal_mask

    # Cross attention mask
    # [B, 1, 1, S] -> [B, 1, T, S]

    cross_mask = src_mask.expand(batch_size, 1, tgt_len, src_len)

    return src_mask, tgt_mask, cross_mask

## 9. Positional Encoding

In [10]:
class Position_Encoding(nn.Module):
    def __init__(self, max_len, d_model):
        """
        max_len: Max Sequence Length/No. of token
        d_model: The dimension of embedding/hidden vector
        """
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        # Create position numbers from 0 to max_len-1 and turn them into a column vector
        position = torch.arange(0, max_len).float().unsqueeze(1)

        # Get even dimension indices
        embedding_index = torch.arange(0, d_model, 2)

        div_term = 1 / torch.tensor(10000) ** (embedding_index / d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Keep the tensor be part of the model, but not is fixed (not learnable) during training
        self.register_buffer('pe', pe)


    def forward(self, embeddings):
        """
        embeddings shape: (batch_size, seq_len, d_model)
        pe shape: (seq_len, d_model)
        """
        return embeddings + self.pe[:embeddings.size(1), :]


"""
   Token: I love cats
Position: 0   1    2

unsqueeze(1)
[[0],
 [1],
 [2]]

embedding_index = torch.arange(0, d_model, 2)

Suppose d_model = 4
embeddnig_index -> [0, 2] (2i)

Sample Usecase:

embeddings =
[
  [0.2, 0.7, 0.1, 0.9],   # I
  [0.4, 0.3, 0.8, 0.2],   # love
  [0.6, 0.5, 0.9, 0.1]    # cats
]

PE =
[
  [0.000, 1.000, 0.000, 1.000],   # position 0
  [0.841, 0.540, 0.010, 1.000],   # position 1
  [0.909,-0.416, 0.020, 1.000]    # position 2
]

embeddings + PE =
[
  [0.200, 1.700, 0.100, 1.900],
  [1.241, 0.840, 0.810, 1.200],
  [1.509, 0.084, 0.920, 1.100]
]
"""

'\n   Token: I love cats\nPosition: 0   1    2\n\nunsqueeze(1)\n[[0],\n [1],\n [2]]\n\nembedding_index = torch.arange(0, d_model, 2)\n\nSuppose d_model = 4\nembeddnig_index -> [0, 2] (2i)\n\nSample Usecase:\n\nembeddings =\n[\n  [0.2, 0.7, 0.1, 0.9],   # I\n  [0.4, 0.3, 0.8, 0.2],   # love\n  [0.6, 0.5, 0.9, 0.1]    # cats\n]\n\nPE =\n[\n  [0.000, 1.000, 0.000, 1.000],   # position 0\n  [0.841, 0.540, 0.010, 1.000],   # position 1\n  [0.909,-0.416, 0.020, 1.000]    # position 2\n]\n\nembeddings + PE =\n[\n  [0.200, 1.700, 0.100, 1.900],\n  [1.241, 0.840, 0.810, 1.200],\n  [1.509, 0.084, 0.920, 1.100]\n]\n'

## 10. Train Tokenizer

In [11]:
# Create corpus
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    with open(CORPUS_FILE, "w", encoding="utf-8") as out:

        for line in f:

            src, tgt = line.rstrip("\n").split("\t")

            out.write(src + "\n")
            out.write(tgt + "\n")

# Train BPE
tokenizer = BPETokenizer(vocab_size=8000)
tokenizer.train([str(CORPUS_FILE)])

# Save
tokenizer.save(str(TOKENIZER_FILE))

print("Tokenizer saved to:", TOKENIZER_FILE)
print("Vocabulary size:", tokenizer.vocab_size)

print("PAD:", tokenizer.pad_id) # 0
print("BOS:", tokenizer.bos_id) # 1
print("EOS:", tokenizer.eos_id) # 2
print("UNK:", tokenizer.unk_id) # 3

Tokenizer saved to: checkpoints/tokenizer.json
Vocabulary size: 8000
PAD: 0
BOS: 1
EOS: 2
UNK: 3


## 11. Transformer Model

In [12]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        """
        d_model = dimension of embeddings
        num_heads = number of attention heads
        d_ff = dimension of inner layer (feed forward neural network)
        """
        super().__init__()

        self.attention = MultiHeadAttenion(d_model, num_heads)
        self.norm1 = LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm2 = LayerNorm(d_model)

    def forward(self, x, mask=None):
        residual = x
        x = self.attention(x, x, x, mask)

        x = x + residual
        x = self.norm1(x)

        residual = x
        x = self.feed_forward(x)

        x = x + residual
        x = self.norm2(x)

        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.self_attention = MultiHeadAttenion(d_model, num_heads)
        self.norm1 = LayerNorm(d_model)

        self.cross_attention = MultiHeadAttenion(d_model, num_heads)
        self.norm2 = LayerNorm(d_model)

        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm3 = LayerNorm(d_model)

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None):
        residual = x
        x = self.self_attention(x, x, x, mask=self_mask)
        x = x + residual
        x = self.norm1(x)

        residual = x
        x = self.cross_attention(x, encoder_output, encoder_output, mask=cross_mask)
        x = x + residual
        x = self.norm2(x)

        residual = x
        x = self.feed_forward(x)
        x = x + residual
        x = self.norm3(x)

        return x

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)

        return x

class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_output, self_mask=None, cross_mask=None):
        for block in self.blocks:
            x = block(x, encoder_output, self_mask, cross_mask)

        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, max_len, d_model, num_heads, d_ff, num_layers):
        """
        vocab_size = number of different tokens
        max_len = maximum sequence length
        d_model = dimension of embeddings
        num_heads = number of attention heads
        d_ff = dimension of inner layer (feed forward neural network)
        num_layer = number of stacked transformer blocks
        """
        super().__init__()

        self.embedding = Token_Embedding(vocab_size, d_model)
        self.position_encoding = Position_Encoding(max_len, d_model)

        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, cross_mask=None):
        # Encoder
        src = self.embedding(src)
        src = self.position_encoding(src)

        encoder_output = self.encoder(src, src_mask)

        # Decoder
        tgt = self.embedding(tgt)
        tgt = self.position_encoding(tgt)

        decoder_output = self.decoder(tgt, encoder_output, self_mask=tgt_mask, cross_mask=cross_mask)

        logits = self.lm_head(decoder_output)

        return logits


## 12. Train Model

In [13]:
from torch.utils.data import DataLoader

# Random Seed to ensure reproducible
set_seed(seed=42)

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

batch_size = 32
d_model = 128
num_heads = 4
d_ff = 512
num_layers = 3
max_len = 256
learning_rate = 3e-4
num_epochs = 20

# Tokenizer
tokenizer = BPETokenizer()
tokenizer.load(str(TOKENIZER_FILE))

vocab_size = tokenizer.vocab_size
print("Vocabulary size:", vocab_size)

# Dataset
dataset = TranslationDataset(str(TRAIN_FILE), tokenizer, max_len)
print("Dataset size:", len(dataset))


# DataLoader
collate_fn = create_collate_fn(tokenizer.pad_id)

loader = DataLoader(dataset, batch_size, shuffle=True, collate_fn=collate_fn)


# Model
model = Transformer(
    vocab_size=vocab_size,
    max_len=max_len,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
)

model = model.to(device)

num_parameters = sum(p.numel() for p in model.parameters())
print(f"Parameters: {num_parameters:,}")


# Loss
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.98), # Beta_1 and Beta_2
    eps=1e-9
)


# Training
for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for src, decoder_input, target in loader:
        src = src.to(device)
        decoder_input = decoder_input.to(device)
        target = target.to(device)

        # Create masks
        src_mask, tgt_mask, cross_mask = create_masks(src, decoder_input, tokenizer.pad_id)

        # Forward pass
        logits = model(src, decoder_input, src_mask, tgt_mask, cross_mask)

        # logits: [B, T, vocab_size]
        # CrossEntropyLoss expects: [N, vocab_size]
        # so flatten B and T.
        B, T, C = logits.shape

        logits = logits.reshape(B * T, C)

        target = target.reshape(B * T)

        # Calculate loss
        loss = criterion(logits, target)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")



# Save model
torch.save(model.state_dict(), str(MODEL_FILE))

print(f"Model saved to {MODEL_FILE}")

Device: cuda
Vocabulary size: 8000
Dataset size: 19990
Parameters: 3,439,936
Epoch 1/20, Loss: 6.2639
Epoch 2/20, Loss: 5.2738
Epoch 3/20, Loss: 4.8460
Epoch 4/20, Loss: 4.5615
Epoch 5/20, Loss: 4.3385
Epoch 6/20, Loss: 4.1566
Epoch 7/20, Loss: 3.9997
Epoch 8/20, Loss: 3.8584
Epoch 9/20, Loss: 3.7296
Epoch 10/20, Loss: 3.6123
Epoch 11/20, Loss: 3.5026
Epoch 12/20, Loss: 3.4036
Epoch 13/20, Loss: 3.3057
Epoch 14/20, Loss: 3.2124
Epoch 15/20, Loss: 3.1279
Epoch 16/20, Loss: 3.0416
Epoch 17/20, Loss: 2.9592
Epoch 18/20, Loss: 2.8822
Epoch 19/20, Loss: 2.8081
Epoch 20/20, Loss: 2.7326
Model saved to checkpoints/transformer.pt


## 13. Translate / Inference

In [14]:
# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d_model = 128
num_heads = 4
d_ff = 512
num_layers = 3
max_len = 256

# Load tokenizer
tokenizer = BPETokenizer()
tokenizer.load(str(TOKENIZER_FILE))

vocab_size = tokenizer.vocab_size


# Load model
model = Transformer(
    vocab_size=vocab_size,
    max_len=max_len,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers
).to(device)

model.load_state_dict(
    torch.load(str(MODEL_FILE), map_location=device)
)

model.eval()



# Create source mask
def create_src_mask(src):

    return (
        (src == tokenizer.pad_id)
        .unsqueeze(1)
        .unsqueeze(2)
    )

# Create target causal mask
def create_tgt_mask(tgt):

    tgt_len = tgt.size(1)

    padding_mask = (
        (tgt == tokenizer.pad_id)
        .unsqueeze(1)
        .unsqueeze(2)
    )

    causal_mask = torch.triu(
        torch.ones(
            tgt_len,
            tgt_len,
            dtype=torch.bool,
            device=tgt.device
        ),
        diagonal=1
    ).unsqueeze(0).unsqueeze(0)

    return padding_mask | causal_mask

# Translation
def translate(text):

    # Tokenize English sentence
    src_ids = tokenizer.encode(text)

    # Convert to tensor
    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    # Source padding mask
    src_mask = create_src_mask(src)

    # Start decoder with BOS
    tgt = torch.tensor(
        [[tokenizer.bos_id]],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        # Encode source sentence
        src_embedding = model.embedding(src)
        src_embedding = model.position_encoding(src_embedding)

        encoder_output = model.encoder(src_embedding, src_mask)

        # Generate tokens one at a time
        for _ in range(max_len - 1):

            tgt_mask = create_tgt_mask(tgt)

            # Embed decoder input
            tgt_embedding = model.embedding(tgt)
            tgt_embedding = model.position_encoding(tgt_embedding)

            # Decode
            decoder_output = model.decoder(tgt_embedding, encoder_output, self_mask=tgt_mask, cross_mask=src_mask)

            # Get prediction for last token
            logits = model.lm_head(decoder_output)

            next_token = torch.argmax(logits[:, -1, :], dim=-1)

            # Add predicted token
            tgt = torch.cat([tgt, next_token.unsqueeze(1)], dim=1)

            # Stop when EOS is generated
            if next_token.item() == tokenizer.eos_id:
                break

    # Convert IDs back to text
    output_ids = tgt[0].tolist()

    # Remove BOS and EOS
    output_ids = [
        token
        for token in output_ids
        if token not in (tokenizer.bos_id, tokenizer.eos_id)
    ]

    return tokenizer.decode(output_ids)


# Test
if __name__ == "__main__":

    text = [
        "I love you.",
        "How are you?",
       " I am happy.",
        "I don't know.",
        "Where are you?",
        "I want to learn French.",
        "She is reading a book.",
        "We are going to Paris.",
        "They live in a small house.",
        "Do you speak English?"
    ]

    for txt in text:
        translation = translate(txt)
        print(f"{txt} : {translation}")

I love you. : Je vous aime.
How are you? : Comment-tu? tu le miens?
 I am happy. : Vite.
I don't know. : Je ne sais pas.
Where are you? : Où êtes-vous?
I want to learn French. : Je voudrais de me rendre.
She is reading a book. : Elle est un livre.
We are going to Paris. : Nous sommes dans Paris.
They live in a small house. : Ils se tonneraient en marche.
Do you speak English? : Que voulez-vous parler?
